# FireSight Data Pipeline

## 1. Imports and paths

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("raw")
OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)
print("RAW:   ", RAW.resolve())
print("OUTPUT:", OUTPUT.resolve())

## 2. Load SIMD raw data

In [ ]:
simd_raw = pd.read_csv(RAW / "simd2020.csv")
print("Shape:", simd_raw.shape)
print("Columns:", simd_raw.columns.tolist())
simd_raw.head()

## 3. Clean Glasgow SIMD data

In [ ]:
# Sanity check: dtypes, nulls, key columns
print(simd_raw.dtypes)
print("\nNull counts:")
print(simd_raw.isnull().sum())
print("\nUnique CA values (top 10):")
print(simd_raw["CA"].value_counts().head(10))

In [ ]:
# Filter to Glasgow City only (Council Area code S12000049)
# WARNING: if the assert below fires, check the CA column output above
GLASGOW_CA = "S12000049"
before = len(simd_raw)
simd_glasgow = simd_raw[simd_raw["CA"] == GLASGOW_CA].copy()
after = len(simd_glasgow)
print(f"Rows before filter: {before}")
print(f"Rows after filter:  {after}")
assert after > 0, f"No rows matched CA == '{GLASGOW_CA}' — check the CA column values above"

In [ ]:
# Rename and select the columns we need
simd_clean = simd_glasgow[["DataZone", "IntZone", "SIMD2020V2CountryDecile"]].copy()
simd_clean.columns = ["zoneid", "zonename", "simddecile"]
print("Shape:", simd_clean.shape)
simd_clean.head()

## 4. Load additional datasets

Three more files will be added to `raw/` as they become available:
- `glasgowdatazones.geojson` — zone boundaries and names
- `fireincidents20192025.csv` — SFRS fire incident records
- `census2022glasgow.csv` — population, age, single-occupancy data

In [ ]:
# --- Census data (add when available) ---
CENSUS_PATH = RAW / "census2022glasgow.csv"
if CENSUS_PATH.exists():
    census = pd.read_csv(CENSUS_PATH)
    print("Census shape:", census.shape)
    print("Census columns:", census.columns.tolist())
    census.head()
else:
    print("census2022glasgow.csv not yet available — skipping")
    census = None

In [ ]:
# --- Fire incident data (add when available) ---
INCIDENTS_PATH = RAW / "fireincidents20192025.csv"
if INCIDENTS_PATH.exists():
    incidents_raw = pd.read_csv(INCIDENTS_PATH)
    print("Incidents shape:", incidents_raw.shape)
    print("Incidents columns:", incidents_raw.columns.tolist())
    incidents_raw.head()
else:
    print("fireincidents20192025.csv not yet available — skipping")
    incidents_raw = None

## 5. Placeholder zone output table

In [ ]:
# Build placeholder table matching the full frontend schema.
# NaN fields will be filled as more data is loaded.
zones = simd_clean.copy()
zones["riskscore"]              = np.nan
zones["drivers"]                = None
zones["population"]             = np.nan
zones["pctover65"]              = np.nan
zones["pctsingleocc"]           = np.nan
zones["forecast6mo"]            = np.nan
zones["forecastlower"]          = np.nan
zones["forecastupper"]          = np.nan
zones["forecast12mo"]           = np.nan
zones["forecast18mo"]           = np.nan
zones["trendpct"]               = np.nan
zones["seasonalitywinterpct"]   = np.nan

SCHEMA_COLS = [
    "zoneid", "zonename", "riskscore", "drivers",
    "population", "pctover65", "pctsingleocc", "simddecile",
    "forecast6mo", "forecastlower", "forecastupper",
    "forecast12mo", "forecast18mo",
    "trendpct", "seasonalitywinterpct"
]
zones = zones[SCHEMA_COLS]
print("Placeholder table shape:", zones.shape)
zones.head()

## 6. Incident data processing (runs when data is available)

In [ ]:
# This cell aggregates raw incidents into 6-month period counts per zone.
# It only runs if incidents_raw was loaded above.

if incidents_raw is not None:
    # --- ADJUST these column names to match the actual CSV ---
    DATE_COL   = "IncidentDate"   # date column in incidents CSV
    ZONE_COL   = "DataZone"       # zone ID column in incidents CSV

    inc = incidents_raw.copy()
    inc[DATE_COL] = pd.to_datetime(inc[DATE_COL])

    # Create a 6-month period label: H1 = Jan-Jun, H2 = Jul-Dec
    inc["period"] = inc[DATE_COL].dt.year.astype(str) + "-H" + (
        (inc[DATE_COL].dt.month > 6).astype(int) + 1
    ).astype(str)

    # Aggregate: count incidents per zone per period
    inc_agg = (
        inc.groupby([ZONE_COL, "period"])
        .size()
        .reset_index(name="incident_count")
    )
    print("Aggregated incident periods shape:", inc_agg.shape)
    print("Sample periods:", inc_agg["period"].unique()[:8])
    inc_agg.head()
else:
    print("No incident data loaded — skipping aggregation")
    inc_agg = None

In [ ]:
# Flag winter periods (Oct-Mar = H2 of year N + H1 of year N+1)
# We tag H2 as winter for seasonality computation.
if inc_agg is not None:
    inc_agg["is_winter"] = inc_agg["period"].str.endswith("H2")
    print("Winter periods:", inc_agg["is_winter"].sum())
    print("Non-winter periods:", (~inc_agg["is_winter"]).sum())
else:
    print("No incident data — skipping winter flag")

## 7. Linear trend forecast (6 / 12 / 18 month)

In [ ]:
def forecast_zone(counts: np.ndarray, n_periods_ahead: list = [1, 2, 3]):
    """
    Fit a linear trend to a 1-D array of 6-month incident counts
    and return forecasts for 1, 2, or 3 periods ahead (6, 12, 18 months).

    Parameters
    ----------
    counts : array of observed 6-month counts, ordered oldest -> newest
    n_periods_ahead : list of integers, each is how many 6-month periods ahead

    Returns
    -------
    dict with keys: forecast (per horizon), lower, upper, trendpct, residual_std
    """
    n = len(counts)
    if n < 2:
        # Not enough history to fit a line — return the last observed count
        last = int(counts[-1]) if n == 1 else 0
        return {h: {"forecast": last, "lower": last, "upper": last} for h in n_periods_ahead}

    x = np.arange(n, dtype=float)
    slope, intercept = np.polyfit(x, counts, 1)

    # Residuals and standard deviation for confidence interval
    fitted = slope * x + intercept
    residuals = counts - fitted
    residual_std = np.std(residuals, ddof=1) if n > 2 else 0.0

    results = {}
    for h in n_periods_ahead:
        raw_forecast = slope * (n - 1 + h) + intercept
        point = max(0.0, raw_forecast)             # clamp negatives to zero
        lower = max(0.0, point - 1.96 * residual_std)
        upper = max(0.0, point + 1.96 * residual_std)
        results[h] = {
            "forecast": round(point),
            "lower":    round(lower),
            "upper":    round(upper),
        }

    # Trend percentage vs the most recent observed period
    latest_observed = counts[-1]
    forecast_6mo = results[1]["forecast"]
    if latest_observed > 0:
        trendpct = round((forecast_6mo - latest_observed) / latest_observed * 100, 1)
    else:
        trendpct = 0.0

    results["trendpct"]     = trendpct
    results["residual_std"] = round(residual_std, 2)
    return results


# --- Quick smoke test with synthetic data ---
test_counts = np.array([4, 5, 6, 7, 8, 9], dtype=float)  # clear upward trend
result = forecast_zone(test_counts, n_periods_ahead=[1, 2, 3])
print("Smoke test — upward trend series:")
print(f"  6-month forecast  (h=1): {result[1]}")
print(f"  12-month forecast (h=2): {result[2]}")
print(f"  18-month forecast (h=3): {result[3]}")
print(f"  trendpct: {result['trendpct']}%")
print()

# Flat series
test_flat = np.array([5, 5, 5, 5, 5, 5], dtype=float)
result_flat = forecast_zone(test_flat)
print("Smoke test — flat series:")
print(f"  6-month forecast: {result_flat[1]}")
print(f"  trendpct: {result_flat['trendpct']}%")
print()

# Declining series (should clamp to 0)
test_declining = np.array([10, 7, 4, 2, 1, 0], dtype=float)
result_dec = forecast_zone(test_declining)
print("Smoke test — declining series (should clamp to 0):")
print(f"  6-month forecast: {result_dec[1]}")
print(f"  trendpct: {result_dec['trendpct']}%")

In [ ]:
# Apply forecasts to all Glasgow zones (runs when incident data is available).
# This cell builds the per-zone time series and calls forecast_zone().

if inc_agg is not None:
    # Pivot to zone x period matrix
    pivot = (
        inc_agg.pivot_table(index=ZONE_COL, columns="period",
                            values="incident_count", fill_value=0)
    )
    # Sort columns chronologically (they are strings like 2019-H1, 2019-H2 ...)
    pivot = pivot.reindex(sorted(pivot.columns), axis=1)
    print("Pivot shape (zones x periods):", pivot.shape)

    forecast_rows = []
    for zoneid, row in pivot.iterrows():
        counts = row.values.astype(float)
        r = forecast_zone(counts, n_periods_ahead=[1, 2, 3])

        # Seasonality: share of incidents in winter (H2) periods
        zone_inc = inc_agg[inc_agg[ZONE_COL] == zoneid]
        total_inc = zone_inc["incident_count"].sum()
        winter_inc = zone_inc.loc[zone_inc["is_winter"], "incident_count"].sum()
        seasonality = round(winter_inc / total_inc * 100, 1) if total_inc > 0 else 0.0

        forecast_rows.append({
            "zoneid":               zoneid,
            "forecast6mo":          r[1]["forecast"],
            "forecastlower":        r[1]["lower"],
            "forecastupper":        r[1]["upper"],
            "forecast12mo":         r[2]["forecast"],
            "forecast18mo":         r[3]["forecast"],
            "trendpct":             r["trendpct"],
            "seasonalitywinterpct": seasonality,
        })

    forecasts_df = pd.DataFrame(forecast_rows)
    print("Forecasts computed for", len(forecasts_df), "zones")
    forecasts_df.head()
else:
    print("No incident data — forecast columns remain NaN in placeholder table")
    forecasts_df = None

In [ ]:
# Merge forecasts back into the zones table (when available)
if forecasts_df is not None:
    zones = zones.drop(columns=[
        "forecast6mo", "forecastlower", "forecastupper",
        "forecast12mo", "forecast18mo",
        "trendpct", "seasonalitywinterpct"
    ])
    zones = zones.merge(forecasts_df, on="zoneid", how="left")
    print("Zones with forecasts merged:", zones.shape)
    print("NaN forecast count:", zones["forecast6mo"].isna().sum())
else:
    print("Skipped — no forecasts to merge")

## 8. Export outputs

In [ ]:
out_path = OUTPUT / "glasgowzones_placeholder.csv"
zones.to_csv(out_path, index=False)
print(f"Saved {len(zones)} zones to {out_path.resolve()}")